# 🧠 Context-Aware Chatbot using LangChain + RAG

**Task 4: Context-Aware Chatbot Using LangChain or RAG**

This notebook builds a conversational chatbot that:

- Remembers conversation history (context memory)
- Retrieves answers from a **vectorized document store** (RAG)
- Uses **LangChain** to orchestrate retrieval + LLM generation
- Is deployed as a live web app using **Streamlit** (via `pyngrok` tunnel, since Colab has no public URL by default)

### How to use this notebook
1. Run the cells top to bottom (Runtime → Run all works, but it's better to go step by step the first time).
2. When prompted, paste your **OpenAI API key** (or swap in another LLM provider — see the note in that cell).
3. Upload your own documents (`.txt`, `.pdf`, `.md`) when prompted, or just use the built-in sample corpus.
4. At the end, the notebook launches a public Streamlit chatbot URL you can open and share.

### Skills covered
Conversational AI • Document embedding & vector search • RAG • LLM integration & deployment


## 1. Install dependencies

In [7]:
!pip -q install \
langchain \
langchain-community \
langchain-huggingface \
langchain-text-splitters \
faiss-cpu \
sentence-transformers \
transformers \
accelerate \
pypdf \
gradio

In [ ]:
# NOTE: If a fresh Colab session shows ModuleNotFoundError on the next cells,
# go to Runtime -> Restart session, then re-run the cells from the top.
print("Dependencies installed. If this is the very first install in this session, restart the runtime now (Runtime -> Restart session), then re-run from the top.")


## Import Libraries

In [1]:
import os

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Set up your API key

This notebook uses **OpenAI** models by default (`gpt-4o-mini` for chat, `text-embedding-3-small` for embeddings)
because it keeps the LangChain code simple and reliable.

> 💡 **Don't have an OpenAI key / want a free alternative?**
> Swap `OpenAIEmbeddings` → `HuggingFaceEmbeddings` (already installed via `sentence-transformers`)
> and `ChatOpenAI` → a HuggingFace pipeline model. Both swaps are called out with comments below.


In [2]:
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"]:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("API key set:", bool(os.environ.get("OPENAI_API_KEY")))


Enter your OpenAI API key: ··········
API key set: True


## 3. Build the knowledge base (custom corpus)

You can either:
- **(A)** Upload your own files (`.txt`, `.pdf`, `.md`) — recommended for a real "internal documents" use case, or
- **(B)** Use the small built-in sample corpus about Anthropic/Claude/LLMs so the notebook runs end-to-end with zero setup.

Run cell (A) if you want to upload files, otherwise just run cell (B).


In [3]:
# --- (A) OPTIONAL: Upload your own documents ---
from google.colab import files
import os

os.makedirs("corpus", exist_ok=True)
print("Upload one or more .txt / .pdf / .md files (or skip this cell to use the sample corpus).")
uploaded = files.upload()

for fname in uploaded:
    dest = os.path.join("corpus", fname)
    with open(dest, "wb") as f:
        f.write(uploaded[fname])

print("Uploaded files saved to ./corpus:", os.listdir("corpus"))


Upload one or more .txt / .pdf / .md files (or skip this cell to use the sample corpus).


Saving ICE KING.pdf to ICE KING (1).pdf
Uploaded files saved to ./corpus: ['ICE KING (1).pdf', 'ICE KING.pdf']


In [ ]:
# --- (B) Sample corpus (skip if you already uploaded files above) ---
import os
os.makedirs("corpus", exist_ok=True)

sample_docs = {
    "llms.txt": '''
Large Language Models (LLMs) are AI systems trained on massive amounts of text
to predict and generate human-like language. They are built on the transformer
architecture, which uses self-attention to weigh the importance of different
words in a sequence. Popular LLM families include GPT, Claude, Llama, and Gemini.

LLMs are typically trained in two stages: pretraining, where the model learns
general language patterns from a large corpus, and fine-tuning (or alignment),
where the model is adjusted using techniques like RLHF (Reinforcement Learning
from Human Feedback) to be more helpful, harmless, and honest.
''',
    "rag.txt": '''
Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval
system with a generative language model. Instead of relying only on knowledge
baked into the model's parameters, RAG first retrieves relevant chunks of text
from an external knowledge base (a vector store) and then feeds those chunks
to the LLM as additional context before it generates an answer.

The typical RAG pipeline has four steps: 1) split documents into chunks,
2) embed each chunk into a vector using an embedding model, 3) store the
vectors in a vector database (e.g., FAISS, Chroma, Pinecone), and 4) at query
time, embed the user's question, retrieve the most similar chunks, and pass
them to the LLM along with the question.

RAG reduces hallucination, keeps answers grounded in real documents, and lets
you update the knowledge base without retraining the model.
''',
    "langchain.txt": '''
LangChain is a framework for building applications powered by language models.
It provides abstractions for chains, agents, memory, and retrievers, making it
easier to combine an LLM with external tools and data sources.

Key LangChain components used in a RAG chatbot include: Document Loaders (to
read files), Text Splitters (to chunk documents), Embeddings (to vectorize
text), Vector Stores (to index and search vectors), Retrievers (to fetch
relevant chunks), and Memory (to keep track of conversation history across
multiple turns so the chatbot has context-aware, multi-turn conversations).
''',
}

for fname, text in sample_docs.items():
    with open(os.path.join("corpus", fname), "w") as f:
        f.write(text.strip())

print("Sample corpus created in ./corpus:", os.listdir("corpus"))


## 4. Load the PDF

In [4]:
import os

# Get the uploaded PDF filename
pdf_file = list(uploaded.keys())[0]

print("PDF:", pdf_file)

loader = PyPDFLoader(pdf_file)
documents = loader.load()

print(f"Loaded {len(documents)} pages.")

PDF: ICE KING (1).pdf
Loaded 52 pages.


## 5. Split the PDF into Chunks

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 36


## 5. Embed documents and build the vector store (FAISS)

This is the "document embedding and vector search" part of the pipeline.
Each chunk is converted into a numeric vector; similar meanings end up close
together in vector space, so we can retrieve relevant chunks for any question.


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings model loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings model loaded.


In [7]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)

vectorstore.save_local("faiss_index")

print("FAISS index created successfully!")

FAISS index created successfully!


## 6. Build the conversational RAG chain (with memory)

`ConversationalRetrievalChain` combines:
- A **retriever** (vector search over our documents)
- An **LLM** (to generate the final answer)
- **Memory** (so the chatbot remembers earlier turns and can handle follow-up
  questions like "what about its downsides?" without repeating context)


In [8]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever created successfully!")

Retriever created successfully!


# 7.Load a Free LLM

In [9]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

pipe = pipeline(
    task="text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=256,
    temperature=0.2,
)

llm = HuggingFacePipeline(pipeline=pipe)

print("LLM loaded successfully!")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM loaded successfully!


# 8.Test the LLM

In [33]:
response = llm.invoke("What is Artificial Intelligence?")

print(response)

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


What is Artificial Intelligence?


## 8.Create the RAG Prompt

In [14]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Use ONLY the information from the provided context to answer the user's question.

If the answer cannot be found in the context, reply:
"I couldn't find that information in the uploaded document."

Context:
{context}

Question:
{input}

Answer:
""")

print("Prompt created successfully!")

Prompt created successfully!


# 9.Build the RAG Chain

In [15]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

document_chain = create_stuff_documents_chain(
    llm,
    prompt
)

rag_chain = create_retrieval_chain(
    retriever,
    document_chain
)

print("RAG chain ready!")

RAG chain ready!


In [16]:
response = rag_chain.invoke(
    {"input": "What is this document about?"}
)

print(response["answer"])

Human: 
You are a helpful AI assistant.

Use ONLY the information from the provided context to answer the user's question.

If the answer cannot be found in the context, reply:
"I couldn't find that information in the uploaded document."

Context:
7 
This is the COMPLETE story of how Frederic Tudor built the world’s first global ice business 
from nothing. 
This timeline is important because your professor will expect: 
• Chronological understanding  
• Entrepreneurial analysis  
• Challenges and solutions  
• Exact business growth journey  
• Failures before success  
• Operational details  
• Innovation details  
• Global expansion process

3. Customers Did Not Understand Ice 
People had never bought imported ice before. 
4. No Storage Infrastructure 
There were no ice houses in the Caribbean. 
5. High Transportation Costs 
Ships were expensive. 
 
Result of First Attempt 
Financial Outcome: 
• Huge loss  
• Very little profit  
• Investors worried  
BUT something very important happ

# 10.Ask Questions

In [17]:
def ask(question):
    response = rag_chain.invoke({"input": question})

    print("=" * 60)
    print("Question:")
    print(question)

    print("\nAnswer:")
    print(response["answer"])

# 11.Test the Chatbot

In [18]:
ask("Summarize the document.")

Question:
Summarize the document.

Answer:
Human: 
You are a helpful AI assistant.

Use ONLY the information from the provided context to answer the user's question.

If the answer cannot be found in the context, reply:
"I couldn't find that information in the uploaded document."

Context:
7 
This is the COMPLETE story of how Frederic Tudor built the world’s first global ice business 
from nothing. 
This timeline is important because your professor will expect: 
• Chronological understanding  
• Entrepreneurial analysis  
• Challenges and solutions  
• Exact business growth journey  
• Failures before success  
• Operational details  
• Innovation details  
• Global expansion process

During Winter: 
1. Lakes and ponds froze.  
2. Workers marked large grids on ice.  
3. Ice cutters sliced giant blocks.  
4. Blocks were pulled by horses.  
5. Ice was stored in insulated warehouses.  
During Shipment: 
6. Ice packed with sawdust.  
7. Loaded onto ships.  
8. Transported worldwide.  
 
10

In [19]:
ask("What are the main topics?")

Question:
What are the main topics?

Answer:
Human: 
You are a helpful AI assistant.

Use ONLY the information from the provided context to answer the user's question.

If the answer cannot be found in the context, reply:
"I couldn't find that information in the uploaded document."

Context:
7 
This is the COMPLETE story of how Frederic Tudor built the world’s first global ice business 
from nothing. 
This timeline is important because your professor will expect: 
• Chronological understanding  
• Entrepreneurial analysis  
• Challenges and solutions  
• Exact business growth journey  
• Failures before success  
• Operational details  
• Innovation details  
• Global expansion process

He pioneered: 
• Cold-chain logistics  
• Global temperature-controlled trade  
• International supply systems  
Modern industries like: 
• Refrigerated shipping  
• Frozen food logistics  
• Cold storage systems  
all follow concepts similar to Tudor’s innovations. 
 
20. MOST IMPORTANT ENTREPRENEURIAL

In [20]:
ask("Who is mentioned in the document?")

Question:
Who is mentioned in the document?

Answer:
Human: 
You are a helpful AI assistant.

Use ONLY the information from the provided context to answer the user's question.

If the answer cannot be found in the context, reply:
"I couldn't find that information in the uploaded document."

Context:
7 
This is the COMPLETE story of how Frederic Tudor built the world’s first global ice business 
from nothing. 
This timeline is important because your professor will expect: 
• Chronological understanding  
• Entrepreneurial analysis  
• Challenges and solutions  
• Exact business growth journey  
• Failures before success  
• Operational details  
• Innovation details  
• Global expansion process

6 
• Born: September 4, 1783  
• Place: Boston

• Caribbean islands  
• Southern United States  
• Cuba  
But every shipment had problems: 
• Melting  
• Delays  
• Poor storage  
• Low efficiency  
 
Financial Disaster 
Because he believed strongly in the idea: 
• He borrowed money heavily.  
•

In [21]:
ask("Who is the president of Mars?")

Question:
Who is the president of Mars?

Answer:
Human: 
You are a helpful AI assistant.

Use ONLY the information from the provided context to answer the user's question.

If the answer cannot be found in the context, reply:
"I couldn't find that information in the uploaded document."

Context:
6 
• Born: September 4, 1783  
• Place: Boston

7

7 
This is the COMPLETE story of how Frederic Tudor built the world’s first global ice business 
from nothing. 
This timeline is important because your professor will expect: 
• Chronological understanding  
• Entrepreneurial analysis  
• Challenges and solutions  
• Exact business growth journey  
• Failures before success  
• Operational details  
• Innovation details  
• Global expansion process

Question:
Who is the president of Mars?

Answer:
Jim Bridenstine

Context:
1 
• Born: December 24, 1964  
• Place: San Antonio, Texas

2

2 
This is the COMPLETE story of how the world’s largest online retailer, Amazon, was founded by Jeff Bezos in 

# 12.Show the Retrieved Source Chunks

In [22]:
response = rag_chain.invoke({"input": "Summarize the document."})

print("Answer:\n")
print(response["answer"])

print("\nRetrieved Chunks:\n")

for i, doc in enumerate(response["context"]):
    print(f"\nChunk {i+1}")
    print("-" * 60)
    print(doc.page_content[:500])

Answer:

Human: 
You are a helpful AI assistant.

Use ONLY the information from the provided context to answer the user's question.

If the answer cannot be found in the context, reply:
"I couldn't find that information in the uploaded document."

Context:
7 
This is the COMPLETE story of how Frederic Tudor built the world’s first global ice business 
from nothing. 
This timeline is important because your professor will expect: 
• Chronological understanding  
• Entrepreneurial analysis  
• Challenges and solutions  
• Exact business growth journey  
• Failures before success  
• Operational details  
• Innovation details  
• Global expansion process

During Winter: 
1. Lakes and ponds froze.  
2. Workers marked large grids on ice.  
3. Ice cutters sliced giant blocks.  
4. Blocks were pulled by horses.  
5. Ice was stored in insulated warehouses.  
During Shipment: 
6. Ice packed with sawdust.  
7. Loaded onto ships.  
8. Transported worldwide.  
 
10. Building the Global Ice Trade

3

# 13.Save the FAISS Index

In [23]:
vectorstore.save_local("faiss_index")

print("FAISS index saved successfully!")

FAISS index saved successfully!


# 14.Add Conversation Memory

In [24]:
!pip -q install langchain-core

In [25]:
chat_history = []

# 15.Create a Context-Aware Chat Function

In [27]:
def ask(question):
    global chat_history

    response = rag_chain.invoke({
        "input": question,
        "chat_history": chat_history
    })

    answer = response["answer"]

    chat_history.append((question, answer))

    return answer

# 16.Test Conversation Memory

In [28]:
print(ask("What is this document about?"))

Human: 
You are a helpful AI assistant.

Use ONLY the information from the provided context to answer the user's question.

If the answer cannot be found in the context, reply:
"I couldn't find that information in the uploaded document."

Context:
7 
This is the COMPLETE story of how Frederic Tudor built the world’s first global ice business 
from nothing. 
This timeline is important because your professor will expect: 
• Chronological understanding  
• Entrepreneurial analysis  
• Challenges and solutions  
• Exact business growth journey  
• Failures before success  
• Operational details  
• Innovation details  
• Global expansion process

3. Customers Did Not Understand Ice 
People had never bought imported ice before. 
4. No Storage Infrastructure 
There were no ice houses in the Caribbean. 
5. High Transportation Costs 
Ships were expensive. 
 
Result of First Attempt 
Financial Outcome: 
• Huge loss  
• Very little profit  
• Investors worried  
BUT something very important happ

In [29]:
print(ask("Can you summarize it?"))

Human: 
You are a helpful AI assistant.

Use ONLY the information from the provided context to answer the user's question.

If the answer cannot be found in the context, reply:
"I couldn't find that information in the uploaded document."

Context:
7 
This is the COMPLETE story of how Frederic Tudor built the world’s first global ice business 
from nothing. 
This timeline is important because your professor will expect: 
• Chronological understanding  
• Entrepreneurial analysis  
• Challenges and solutions  
• Exact business growth journey  
• Failures before success  
• Operational details  
• Innovation details  
• Global expansion process

Invested heavily despite uncertainty. 
Problem Solving 
Solved transportation and melting challenges. 
Long-Term Thinking 
Focused on building an entire industry. 
 
18. Weaknesses and Criticism 
Not everything about Tudor was perfect. 
Criticisms 
Heavy Debt 
He often borrowed excessively. 
Aggressive Business Behavior 
Some people considered him

In [30]:
print(ask("Tell me more about that."))

Human: 
You are a helpful AI assistant.

Use ONLY the information from the provided context to answer the user's question.

If the answer cannot be found in the context, reply:
"I couldn't find that information in the uploaded document."

Context:
3. Customers Did Not Understand Ice 
People had never bought imported ice before. 
4. No Storage Infrastructure 
There were no ice houses in the Caribbean. 
5. High Transportation Costs 
Ships were expensive. 
 
Result of First Attempt 
Financial Outcome: 
• Huge loss  
• Very little profit  
• Investors worried  
BUT something very important happened: 
People Loved Ice. 
Rich customers became excited about: 
• Cold drinks  
• Cooling food  
• Luxury lifestyle  
Tudor realized: 
“The idea works. The system is the problem. ” 
This is a key entrepreneurial lesson. 
 
4. Years of Failure and Debt (1807–1815) 
Repeated Attempts 
Tudor continued shipping ice despite losses. 
He exported ice to:

• Sent to Martinique  
Result: 
• Much of the ice me

# 17.Create a Gradio Interface

In [34]:
import gradio as gr

def chatbot(message, history):
    response = ask(message)
    return response

demo = gr.ChatInterface(
    fn=chatbot,
    title="📄 Context-Aware RAG Chatbot",
    description="Upload a PDF and ask questions about it."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e4a3b6f41d208bd069.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# 18.Save the FAISS Index

In [35]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("FAISS index loaded successfully!")

FAISS index loaded successfully!
